In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.covariance import LedoitWolf

## 1. Loading the Data

Load input data from pickle file containing price, returns, volume, and other market data.

In [2]:
with open("hw3_input.pickle", "rb") as f:
    data = pickle.load(f)

In [3]:
price = data["price"]
tri = data["tri"]
volume = data["volume"]
mtbv = data["mtbv"]
cap = data["cap"]
tcost = data["tcost"]
rec = data["rec"]
isactivenow = data["isactive"]

allstocks = price.columns.to_numpy()
myday = price.index.to_numpy()

T, n = price.shape
print(f"T = {T}, n = {n}")

T = 1521, n = 411


## 2. Risk Model

Build monthly covariance matrices using Ledoit-Wolf shrinkage estimator with 252-day rolling windows.

In [50]:
returns = tri.apply(pd.to_numeric, errors="coerce") / 100.0
returns = returns.fillna(0)

all_dates = returns.index

In [53]:
year_month_list = []
for dt in all_dates:
    ym = (dt.year, dt.month)
    if ym not in year_month_list:
        year_month_list.append(ym)

shrink_list = []
month_ref_dates = []
cov_by_month = {}

In [54]:
def get_month_reference_date(year, month, date_index):
    first_calendar_day = pd.Timestamp(year=year, month=month, day=1)
    mask = date_index <= first_calendar_day
    if not mask.any():
        return None
    ref_date = date_index[mask].max()
    return ref_date

In [55]:
for (year, month) in year_month_list:
    ref_date = get_month_reference_date(year, month, all_dates)
    if ref_date is None:
        continue
    
    try:
        ref_pos = all_dates.get_loc(ref_date)
    except KeyError:
        continue
    
    if ref_pos < 252:
        continue
    
    start_pos = ref_pos - 252
    end_pos = ref_pos
    
    returns_window = returns.iloc[start_pos:end_pos]
    
    active_row = isactivenow.loc[ref_date]
    active_mask = (active_row == 1)
    active_stocks = list(isactivenow.columns[active_mask])
    
    if len(active_stocks) == 0:
        continue
    
    R = returns_window[active_stocks]
    R_filled = R.fillna(0)
    
    X = R_filled.values
    lw = LedoitWolf(assume_centered=False)
    lw.fit(X)
    cov_matrix = lw.covariance_
    alpha = float(lw.shrinkage_)
    
    if alpha < 0:
        alpha = 0.0
    
    shrink_list.append(alpha)
    month_ref_dates.append(ref_date)
    
    cov_df = pd.DataFrame(
        cov_matrix,
        index=active_stocks,
        columns=active_stocks
    )
    cov_by_month[ref_date] = cov_df


In [56]:
shrink = np.array(shrink_list).reshape(-1, 1)

print("Number of months with risk model:", shrink.shape[0])

Number of months with risk model: 59


## 3. Alphas

Generate and process alpha signals by demeaning, standardizing, and winsorizing for active stocks only.

In [57]:
def process_alpha(alpha_raw, isactivenow, winsor_limit=3.0):
    alpha = alpha_raw.copy().astype(float)
    
    for date in alpha.index:
        active_mask = (isactivenow.loc[date] == 1)
        if active_mask.sum() == 0:
            alpha.loc[date, :] = 0.0
            continue
        
        vals = alpha.loc[date, active_mask]
        mean_val = vals.mean()
        std_val = vals.std()
        
        if (std_val is None) or (std_val == 0) or np.isnan(std_val):
            alpha.loc[date, :] = 0.0
            continue
        
        z = (vals - mean_val) / std_val
        z = z.clip(lower=-winsor_limit, upper=winsor_limit)
        alpha.loc[date, active_mask] = z
        alpha.loc[date, ~active_mask] = 0.0
    
    alpha = alpha.fillna(0.0)
    return alpha

### SHORT-TERM CONTRARIAN

Compute contrarian alpha using weighted sum of past returns with triangular weights, negated for mean reversion.

In [58]:
K_REV = 10

alpharev_raw = pd.DataFrame(0.0, index=all_dates, columns=allstocks)

weights = list(range(1, K_REV + 1))
weights = weights[::-1]

for t in range(K_REV, T):
    window = returns.iloc[t-K_REV:t]
    weighted_sum = pd.Series(0.0, index=allstocks)
    
    for k in range(K_REV):
        day_returns = window.iloc[k]
        weight = weights[k]
        weighted_sum = weighted_sum + weight * day_returns
    
    alpharev_raw.iloc[t] = -weighted_sum

alpharev = process_alpha(alpharev_raw, isactivenow)
alpharev

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.068885,1.490303,-2.355891,-0.449095,0.0,0.416289,-0.584330,0.313190,0.122747,0.0,...,0.0,-0.313453,-0.335672,0.949907,0.115154,-3.000000,0.122142,-0.052000,1.226175,0.085953
2025-07-29,0.115786,1.256422,-1.982942,-0.144660,0.0,-0.103408,-0.172701,0.415336,0.495998,0.0,...,0.0,-0.124276,-0.166683,0.124949,0.408502,-3.000000,1.177869,0.115765,-0.141895,0.001185
2025-07-30,0.147823,1.674313,-1.687536,-0.232517,0.0,-0.243042,-0.340345,0.127067,0.262458,0.0,...,0.0,-0.275416,-0.299015,0.356953,0.446379,-1.968427,1.251299,0.008449,0.201976,0.316049


### SHORT-TERM PROCYCLICAL

Generate momentum alpha from analyst recommendation revisions over 20-day lookback period.

In [59]:
H_REC = 20

alpharec_raw = rec - rec.shift(H_REC)
alpharec_raw = alpharec_raw.fillna(0)

alpharec = process_alpha(alpharec_raw, isactivenow)
alpharec

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.101798,-0.101798,-0.101798,-0.101798,0.0,1.877615,-1.091505,-0.101798,-0.101798,0.0,...,0.0,-1.091505,-0.101798,-1.091505,0.887908,0.887908,-0.101798,-0.101798,0.887908,-2.081212
2025-07-29,-0.107566,-0.107566,-0.107566,-0.107566,0.0,1.927464,-1.125081,-0.107566,-0.107566,0.0,...,0.0,-1.125081,-0.107566,-0.107566,0.909949,0.909949,-0.107566,-0.107566,0.909949,-2.142595
2025-07-30,-0.133384,-0.133384,-0.133384,-0.133384,0.0,1.935547,-1.167849,-0.133384,-0.133384,0.0,...,0.0,-1.167849,-0.133384,-0.133384,0.901081,0.901081,-0.133384,-0.133384,0.901081,-2.202314


### LONG-TERM CONTRARIAN

Create value alpha by negating market-to-book ratio to identify undervalued stocks for long-term contrarian strategy.

In [60]:
alphaval_raw = -mtbv.copy().astype(float)

alphaval_raw = alphaval_raw.fillna(0)

alphaval = process_alpha(alphaval_raw, isactivenow)
alphaval

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-29,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-30,0.205700,0.168883,0.622919,0.077622,0.0,-1.792137,0.454675,0.025840,0.285338,0.0,...,0.0,-0.208879,-0.713995,-2.213715,0.712331,0.531899,-0.550509,-0.147500,0.478032,0.600817


### LONG-TERM PROCYCLICAL

Compute momentum alpha from 12-month total return index, skipping most recent month to avoid microstructure effects.

In [61]:
SKIP_1M = 21
LOOKBACK_12M = 252

tri_1m_ago = tri_clean.shift(SKIP_1M)
tri_12m_ago = tri_clean.shift(LOOKBACK_12M)

alphamom_raw = (tri_1m_ago / tri_12m_ago) - 1
alphamom_raw = alphamom_raw.replace([np.inf, -np.inf], np.nan)
alphamom_raw = alphamom_raw.fillna(0.0)

alphamom = process_alpha(alphamom_raw, isactivenow)
alphamom

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.051704,-0.098601,-0.078957,0.122306,0.0,0.109639,0.274946,0.083713,-0.159285,0.0,...,0.0,0.651695,0.646750,0.279946,-0.093449,-0.063485,-0.046062,0.040743,0.164973,-0.034679
2025-07-29,-0.173063,0.213880,0.005538,-0.250725,0.0,0.011196,-0.109731,0.097634,0.286369,0.0,...,0.0,0.259841,-0.816989,-1.449810,0.966092,0.001659,-0.058754,-0.401767,-0.319902,0.227780
2025-07-30,0.042158,0.042099,0.160982,-0.006243,0.0,0.055492,-0.010200,0.063182,-0.001575,0.0,...,0.0,0.031610,-0.052049,-0.034930,0.239062,0.121451,0.118930,0.006784,-0.061160,0.002558


### BLEND

Combine individual alpha signals using weighted linear combination with weights 50% reversal, 25% recommendation, 15% value, 10% momentum.

In [62]:
alphablend_raw = (
    0.50 * alpharev +
    0.25 * alpharec +
    0.15 * alphaval +
    0.10 * alphamom
)

alphablend = process_alpha(alphablend_raw, isactivenow)
alphablend

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.065866,1.266653,-1.957313,-0.405917,0.0,0.717335,-0.829018,0.236989,0.096814,0.0,...,0.0,-0.701135,-0.422764,-0.190415,0.643687,-2.108239,-0.102331,-0.133458,1.593978,-0.692103
2025-07-29,0.062271,1.136993,-1.663147,-0.217708,0.0,0.271822,-0.569777,0.329123,0.503932,0.0,...,0.0,-0.638239,-0.549795,-0.803756,1.114238,-2.140777,0.826169,-0.072902,0.332494,-0.768779
2025-07-30,0.125941,1.505786,-1.409600,-0.264169,0.0,0.168659,-0.731030,0.061740,0.244109,0.0,...,0.0,-0.845551,-0.549295,-0.358562,1.042987,-1.226466,0.938103,-0.103885,0.701779,-0.562076


## 4 Optimizer

Define trading window and helper function to retrieve appropriate covariance matrix for portfolio optimization.

In [63]:
start_date = pd.Timestamp("2021-01-01")
end_date   = pd.Timestamp("2025-08-01")

dates = returns.index

start_idx = int(np.where(dates >= start_date)[0][0])
end_idx   = int(np.where(dates <= end_date)[0][-1])

t0 = start_idx
print("t0 index =", t0, "date =", dates[t0].date(), "end date =", dates[end_idx].date())


t0 index = 343 date = 2021-01-04 end date = 2025-08-01


In [64]:
month_ref_dates = pd.to_datetime(month_ref_dates)
month_ref_dates_sorted = month_ref_dates.sort_values()

def get_cov_for_date(current_date: pd.Timestamp):
    mask = month_ref_dates_sorted <= current_date
    if not mask.any():
        return None, None
    ref_date = month_ref_dates_sorted[mask].max()
    Sigma_df = cov_by_month[ref_date]
    active_stocks = Sigma_df.index.to_numpy()
    return Sigma_df, active_stocks


In [84]:
returns_np = price.pct_change().fillna(0.0).clip(-1, 1).to_numpy(dtype=float)
alphablend_np = alphablend.fillna(0.0).to_numpy(dtype=float)

tcost_df = tcost.copy()
tcost_df = tcost_df.where(isactivenow == 1)  # inactive -> NaN
tcost_df = tcost_df.clip(lower=0.0, upper=0.01)
tcost_df = tcost_df.apply(lambda row: row.fillna(row.median()), axis=1)

tcost_np = tcost_df.to_numpy()

active_numeric = (
    isactivenow
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)
    .fillna(0.0)
)
active_np = (active_numeric.to_numpy(dtype=float) > 0.5)

T, n = returns_np.shape


/var/folders/91/sp6w_bs519qdrbs4z7ny5xx80000gn/T/ipykernel_552/3002959400.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_np = price.pct_change().fillna(0.0).clip(-1, 1).to_numpy(dtype=float)
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/var/folders/91/sp6w_bs519qdrbs4z7ny5xx80000gn/T/ipykernel_552/3002959400.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tcost_df = tcost_df.apply(lambda row: row.fillna(row.median()), axis=1)
/opt/an

In [117]:
def build_target(alpha_vec: np.ndarray,
                 Sigma_df: pd.DataFrame,
                 active_mask: np.ndarray,
                 mu_param: float) -> np.ndarray:
    target = np.zeros_like(alpha_vec)

    stocks_in_cov = Sigma_df.index.to_numpy()
    idx_cov = pd.Index(allstocks).get_indexer(stocks_in_cov)
    mask_active_cov = active_mask[idx_cov]

    if mask_active_cov.sum() < 2:
        return target

    idx = idx_cov[mask_active_cov]
    alpha_sub = alpha_vec[idx]
    Sigma_sub = Sigma_df.loc[stocks_in_cov[mask_active_cov],
                             stocks_in_cov[mask_active_cov]].to_numpy(dtype=float)

    Sigma_sub = Sigma_sub + 1e-6 * np.eye(Sigma_sub.shape[0])

    try:
        w_dir = np.linalg.solve(Sigma_sub, alpha_sub)
    except np.linalg.LinAlgError:
        w_dir = np.linalg.pinv(Sigma_sub) @ alpha_sub

    if (w_dir > 0).all() or (w_dir < 0).all():
        order = np.argsort(alpha_sub)
        k = len(order) // 2
        long_idx = order[k:]
        short_idx = order[:k]
        w_dir = np.zeros_like(alpha_sub)
        w_dir[long_idx] = alpha_sub[long_idx]
        w_dir[short_idx] = -alpha_sub[short_idx]

    pos_sum = w_dir[w_dir > 0].sum()
    w_dir[w_dir > 0] = w_dir[w_dir > 0] / pos_sum
    neg_sum = -w_dir[w_dir < 0].sum()
    w_dir[w_dir < 0] = w_dir[w_dir < 0] / neg_sum
    w_dir = np.clip(w_dir, -0.1, 0.1)

    w_sub = mu_param / w_dir
    target[idx] = w_sub
    return target


## 5. Backtest

Run backtest simulation with partial execution, tracking positions, trades, PnL, and portfolio metrics over time.

In [118]:
def run_backtest(lambda_param: float, mu_param: float, capture_full: bool = True):
    lambda_param = float(np.clip(lambda_param, 0.0, 1.0))
    mu_param     = float(mu_param)

    positions = np.zeros((T, n))
    trade     = np.zeros((T, n))
    pnl       = np.zeros(T)
    daily_pnl = np.zeros(T)
    booksize  = np.zeros(T)
    tradesize = np.zeros(T)
    long_side  = np.zeros(T)
    short_side = np.zeros(T)

    for t in range(1, T):
        date_t   = dates[t]
        date_t_1 = dates[t-1]

        pretrade_positions = positions[t-1]
        gross_pnl_t = float(np.dot(positions[t-1], returns_np[t]))

        if (date_t < start_date) or (date_t > end_date):
            positions[t]  = pretrade_positions
            daily_pnl[t]  = gross_pnl_t
            pnl[t]        = pnl[t-1] + gross_pnl_t
            booksize[t]   = np.abs(positions[t]).sum()
            tradesize[t]  = 0.0
            long_side[t]  = positions[t][positions[t] > 0].sum()
            short_side[t] = -positions[t][positions[t] < 0].sum()
            continue

        Sigma_df, _ = get_cov_for_date(date_t_1)
        if Sigma_df is None:
            target = positions[t-1].copy()
        else:
            alpha_vec   = alphablend_np[t-1]
            active_mask = active_np[t-1]
            target = build_target(alpha_vec, Sigma_df, active_mask, mu_param)
            target[~active_mask] = 0.0


        desired_delta = target - positions[t-1]
        trade_t = lambda_param * desired_delta

        cost_t = np.abs(trade_t) * tcost_np[t]
        cost_t_sum = float(cost_t.sum())

        net_pnl_t = gross_pnl_t - cost_t_sum
        daily_pnl[t] = net_pnl_t
        pnl[t] = pnl[t-1] + net_pnl_t

        positions[t] = pretrade_positions + trade_t
        trade[t] = trade_t

        booksize[t]  = np.abs(positions[t]).sum()
        tradesize[t] = np.abs(trade_t).sum()
        long_side[t]  = positions[t][positions[t] > 0].sum()
        short_side[t] = -positions[t][positions[t] < 0].sum()

    window = slice(start_idx, end_idx + 1)
    trade_window = tradesize[window]
    if trade_window.size > 1:
        avg_trade_ex_first = trade_window[1:].mean()
    elif trade_window.size == 1:
        avg_trade_ex_first = trade_window[0]
    else:
        avg_trade_ex_first = 0.0

    avg_long  = long_side[window].mean() if trade_window.size > 0 else 0.0
    avg_short = short_side[window].mean() if trade_window.size > 0 else 0.0

    result = {
        "lambda": lambda_param,
        "mu": mu_param,
        "avg_trade_ex_first": avg_trade_ex_first,
        "avg_long": avg_long,
        "avg_short": avg_short,
        "daily_pnl": pd.Series(daily_pnl, index=dates, name="daily_pnl"),
        "pnl": pd.Series(pnl, index=dates, name="pnl"),
        "booksize": pd.Series(booksize, index=dates, name="booksize"),
        "tradesize": pd.Series(tradesize, index=dates, name="tradesize"),
    }

    if capture_full:
        result["trade"] = pd.DataFrame(trade, index=dates, columns=allstocks)
        result["back_weight"] = pd.DataFrame(positions, index=dates, columns=allstocks)

    return result


In [119]:
lambda_candidates = np.linspace(0.25, 0.65, 5)
mu_candidates = [10000000, 15000000, 20000000]

search_rows = []
best_combo = None
best_score = np.inf

for lam in lambda_candidates:
    for mu in mu_candidates:
        res = run_backtest(lam, mu, capture_full=False)
        avg_trade = res["avg_trade_ex_first"]
        avg_long  = res["avg_long"]
        avg_short = res["avg_short"]

        score = (
            ((avg_trade - 15000000) / 15000000) ** 2
            + ((avg_long  - 50000000) / 50000000) ** 2
            + ((avg_short - 50000000) / 50000000) ** 2
        )

        search_rows.append({
            "lambda": lam,
            "mu": mu,
            "avg_trade_ex_first": avg_trade,
            "avg_long": avg_long,
            "avg_short": avg_short,
            "score": score,
        })

        if score < best_score:
            best_score = score
            best_combo = (lam, mu)

search_df = pd.DataFrame(search_rows).sort_values("score").reset_index(drop=True)
search_df.head(10)


,lambda,mu,avg_trade_ex_first,avg_long,avg_short,score
0,0.25,10000000,3.179882e+12,4.324735e+12,2.052728e+12,5.410680e+10
1,0.35,10000000,4.479031e+12,4.401346e+12,2.128747e+12,9.872370e+10
2,0.25,15000000,4.769823e+12,6.487103e+12,3.079091e+12,1.217408e+11
3,0.45,10000000,5.788074e+12,4.462951e+12,2.189850e+12,1.587812e+11
4,0.25,20000000,6.359764e+12,8.649471e+12,4.105455e+12,2.164286e+11
5,0.35,15000000,6.718547e+12,6.602019e+12,3.193121e+12,2.221290e+11
6,0.55,10000000,7.107146e+12,4.516372e+12,2.242850e+12,2.346656e+11
7,0.65,10000000,8.436999e+12,4.564823e+12,2.290944e+12,3.268017e+11
8,0.45,15000000,8.682111e+12,6.694426e+12,3.284774e+12,3.572585e+11
9,0.35,20000000,8.958062e+12,8.802692e+12,4.257494e+12,3.948965e+11


In [120]:
print(f"Chosen lambda = {best_combo[0]:.3f}, mu = {best_combo[1]/1e6:.1f}M")

final_results = run_backtest(best_combo[0], best_combo[1], capture_full=True)

trade       = final_results["trade"]
back_weight = final_results["back_weight"]
pnl         = final_results["pnl"]
booksize    = final_results["booksize"]
tradesize   = final_results["tradesize"]
daily_pnl   = final_results["daily_pnl"]

lambda_value = final_results["lambda"]
mu_value     = final_results["mu"]

lambda_scalar = np.array([[lambda_value]])
mu_scalar     = np.array([[mu_value]])

print(f"Average daily trade (excluding first day): {final_results['avg_trade_ex_first']/1e6:.2f}M")
print(f"Average long book: {final_results['avg_long']/1e6:.2f}M")
print(f"Average short book: {final_results['avg_short']/1e6:.2f}M")



Chosen lambda = 0.250, mu = 10.0M
Average daily trade (excluding first day): 3179882.10M
Average long book: 4324735.40M
Average short book: 2052727.64M


## 6. Evaluation

Calculate performance metrics including annualized Sharpe ratio, maximum drawdown, and longest drawdown period.

In [121]:
daily_pnl_nonzero = daily_pnl.iloc[1:]
pnl_std = daily_pnl_nonzero.std()

if pnl_std > 0:
    sharpe = float(np.sqrt(252) * daily_pnl_nonzero.mean() / pnl_std)
else:
    sharpe = 0.0

print("Annualized Sharpe =", sharpe)

cum_pnl = pnl
hwm = cum_pnl.cummax()
drawdown = hwm - cum_pnl

deepest_dd = float(drawdown.max())

drawdown_flags = drawdown > 1e-8
longest_dd = 0
current_run = 0
for flag in drawdown_flags:
    if flag:
        current_run += 1
        longest_dd = max(longest_dd, current_run)
    else:
        current_run = 0

print("Deepest drawdown =", deepest_dd)
print("Longest drawdown (days) =", longest_dd)

sharpe_val     = np.array([[sharpe]])
longest_dd_val = np.array([[longest_dd]])
deepest_dd_val = np.array([[deepest_dd]])


Annualized Sharpe = -0.1987568630604556
Deepest drawdown = 1444227699209.715
Longest drawdown (days) = 163


In [122]:
def series_to_T1(s):
    return s.to_numpy(dtype=float).reshape(-1, 1)

def scalar_to_1x1(x):
    return np.array([[float(x)]], dtype=float)

shrink_out = np.asarray(shrink, dtype=float).reshape(-1, 1)

alpharev_out   = alpharev.to_numpy(dtype=float)
alpharec_out   = alpharec.to_numpy(dtype=float)
alphaval_out   = alphaval.to_numpy(dtype=float)
alphamom_out   = alphamom.to_numpy(dtype=float)
alphablend_out = alphablend.to_numpy(dtype=float)

lambda_out    = scalar_to_1x1(lambda_value)
mu_out        = scalar_to_1x1(mu_value)
t0_out        = scalar_to_1x1(t0)
sharpe_out    = scalar_to_1x1(sharpe)
longest_dd_out = scalar_to_1x1(longest_dd)
deepest_dd_out = scalar_to_1x1(deepest_dd)

pnl_out        = series_to_T1(pnl)
booksize_out   = series_to_T1(booksize)
tradesize_out  = series_to_T1(tradesize)

trade_out       = trade.to_numpy(dtype=float)
back_weight_out = back_weight.to_numpy(dtype=float)

results = {
    "shrink":      shrink_out,
    "alpharev":    alpharev_out,
    "alpharec":    alpharec_out,
    "alphaval":    alphaval_out,
    "alphamom":    alphamom_out,
    "alphablend":  alphablend_out,
    "lambda":      lambda_out,
    "mu":          mu_out,
    "t0":          t0_out,
    "trade":       trade_out,
    "back_weight": back_weight_out,
    "pnl":         pnl_out,
    "booksize":    booksize_out,
    "tradesize":   tradesize_out,
    "sharpe":      sharpe_out,
    "longest_dd":  longest_dd_out,
    "deepest_dd":  deepest_dd_out,
}

output_path = "ps3_output.pkl"
with open(output_path, "wb") as f:
    pickle.dump(results, f)

print("Saved problem set output to:", output_path)


Saved problem set output to: ps3_output.pkl
